# 2. Process data

In this notebook I calculate the Normalised Burn Ratio (NBR) and the change in NBR since before the fire (dNBR) for each Landsat image.

In [2]:
from pathlib import Path

import geopandas as gpd
import pandas as pd
import rioxarray as rxr
import xarray as xr

## File paths

In [3]:
data_dir = Path("..") / "data"
multispectral_dir = data_dir / "multispectral"
nbr_dir = data_dir / "nbr"
dnbr_dir = data_dir / "dnbr"
timeseries_dir = data_dir / "timeseries"
fire_outline_fpath = data_dir / "effis_modis_powys_20260427.gpkg"

for directory in [nbr_dir, dnbr_dir, timeseries_dir]:
    directory.mkdir(parents=True, exist_ok=True)

aoi_name = "powys"

## Load the multispectral images as a datacube

In [4]:
filepaths = sorted(multispectral_dir.glob(f"{aoi_name}_*.tif"))

scenes_list = []
dates = []

for filepath in filepaths:
    scene = rxr.open_rasterio(filepath)
    scene["band"] = list(scene.attrs["long_name"])  # restore the band names
    scenes_list.append(scene)

    # The date is the second part of the filename, e.g. "powys_2026-04-22_landsat-9"
    dates.append(filepath.stem.split("_")[1])

scenes = xr.concat(scenes_list, dim="time")
scenes["time"] = pd.to_datetime(dates)

scenes

<xarray.DataArray (time: 4, band: 6, y: 570, x: 500)> Size: 27MB
array([[[[0.0181575 , 0.01840501, 0.0229425 , ..., 0.0442275 ,
          0.04378749, 0.04356749],
         [0.020165  , 0.02112749, 0.0229975 , ..., 0.044475  ,
          0.0451625 , 0.04290751],
         [0.02209   , 0.0221725 , 0.0287175 , ..., 0.04274251,
          0.0399375 , 0.0421375 ],
         ...,
         [0.03111   , 0.03204499, 0.02720501, ..., 0.07285501,
          0.06240501, 0.0505525 ],
         [0.0329525 , 0.030725  , 0.0317425 , ..., 0.07359751,
          0.07169999, 0.05462249],
         [0.03600501, 0.03443749, 0.0372425 , ..., 0.06958251,
          0.06930749, 0.05643751]],

        [[0.03660999, 0.03817751, 0.04068001, ..., 0.0806375 ,
          0.08256249, 0.0815175 ],
         [0.0396625 , 0.0411475 , 0.04166999, ..., 0.08272751,
          0.08492748, 0.08077501],
         [0.04345749, 0.0442275 , 0.05343999, ..., 0.08036251,
          0.07799749, 0.077805  ],
...
          0.2637875 , 0.25295252],
         [0.194405  , 0.201555  , 0.1955875 , ..., 0.28672248,
          0.29263502, 0.2503125 ],
         [0.1828825 , 0.20447   , 0.2018025 , ..., 0.28672248,
          0.28947252, 0.26928753]],

        [[       nan,        nan,        nan, ..., 0.10283001,
          0.10074   , 0.09826501],
         [       nan,        nan,        nan, ..., 0.0994475 ,
          0.10071249, 0.09892501],
         [       nan,        nan,        nan, ..., 0.10101499,
          0.09680749, 0.10041   ],
         ...,
         [0.0907575 , 0.10230751, 0.0838825 , ..., 0.16805999,
          0.1533475 , 0.14273249],
         [0.09438749, 0.09969501, 0.09425001, ..., 0.158655  ,
          0.1629725 , 0.1417975 ],
         [0.09950249, 0.10731249, 0.10822   , ..., 0.155795  ,
          0.1560425 , 0.14749001]]]],
      shape=(4, 6, 570, 500), dtype=float32)
Coordinates:
  * time         (time) datetime64[ns] 32B 2026-04-22 2026-04-29 ... 2026-05-24
  * band         (band) <U6 144B 'blue' 'green' 'red' 'nir08' 'swir16' 'swir22'
  * y            (y) float64 5kB 5.804e+06 5.804e+06 ... 5.787e+06 5.787e+06
  * x            (x) float64 4kB 4.452e+05 4.453e+05 ... 4.602e+05 4.602e+05
    spatial_ref  int64 8B 0
Attributes:
    AREA_OR_POINT:  Area
    scale_factor:   1.0
    add_offset:     0.0
    long_name:      ('blue', 'green', 'red', 'nir08', 'swir16', 'swir22')

Load fire outline

In [5]:
fire_gdf = gpd.read_file(fire_outline_fpath).to_crs(scenes.rio.crs)

## Normalised Burn Ratio (NBR)

NBR uses the near-infrared (NIR) and the second shortwave-infrared (SWIR2) bands. Healthy vegetation reflects strongly in the NIR and weakly in the SWIR, whereas burned areas show the opposite, so NBR drops sharply after a fire:

$$NBR = \frac{NIR - SWIR2}{NIR + SWIR2}$$

In [6]:
nir = scenes.sel(band="nir08")
swir2 = scenes.sel(band="swir22")

nbr = (nir - swir2) / (nir + swir2)
nbr.name = "nbr"
nbr.attrs = {"long_name": "NBR"}  # replace the attributes inherited from the multispectral data

In [8]:
for t in nbr["time"].values:
    date = str(t)[:10]
    filepath = nbr_dir / f"{aoi_name}_{date}_nbr.tif"
    nbr.sel(time=t).rio.to_raster(filepath, compress="deflate", dtype="float32")
    print(f"Saved {filepath}")

Saved ../data/nbr/powys_2026-04-22_nbr.tif
Saved ../data/nbr/powys_2026-04-29_nbr.tif
Saved ../data/nbr/powys_2026-04-30_nbr.tif
Saved ../data/nbr/powys_2026-05-24_nbr.tif


## Change in NBR (dNBR)

dNBR is the difference between the pre-fire NBR and the NBR on each date:

$$dNBR = NBR_{prefire} - NBR_{postfire}$$

My first image (22 April) was acquired before the fire, so I use it as the pre-fire reference. Positive dNBR values indicate a loss of vegetation.

In [9]:
nbr_prefire = nbr.isel(time=0)
dnbr = nbr_prefire - nbr
dnbr.name = "dnbr"
dnbr.attrs = {"long_name": "dNBR"}

for t in dnbr["time"].values:
    date = str(t)[:10]
    filepath = dnbr_dir / f"{aoi_name}_{date}_dnbr.tif"
    dnbr.sel(time=t).rio.to_raster(filepath, compress="deflate", dtype="float32")
    print(f"Saved {filepath}")

Saved ../data/dnbr/powys_2026-04-22_dnbr.tif
Saved ../data/dnbr/powys_2026-04-29_dnbr.tif
Saved ../data/dnbr/powys_2026-04-30_dnbr.tif
Saved ../data/dnbr/powys_2026-05-24_dnbr.tif


## Burned area through time

To estimate the burned area, I threshold dNBR using the burn severity classes recommended by [UN-SPIDER](https://un-spider.org/advisory-support/recommended-practices/recommended-practice-burn-severity/in-detail/normalized-burn-ratio). I count pixels with dNBR above 0.27, which corresponds to at least "low-moderate" burn severity. To test how sensitive my results are to this choice, I repeat the calculation with thresholds of 0.26 and 0.28.

The function allows us to repeat the same calculation for every date and every threshold.

In [10]:
def calculate_thresholded_area(da, threshold, pixel_area, threshold_type="greater"):
    """
    Calculate the total area of pixels in a 2D DataArray above or below a threshold.

    Parameters
    ----------
    da : xarray.DataArray
        2D array with dimensions (y, x).
    threshold : float
        Threshold value to compare against.
    pixel_area : float
        Area of a single pixel (the output will be in the same units).
    threshold_type : str
        "greater" to count pixels above the threshold, or "less" to count pixels below it.

    Returns
    -------
    float
        Total area of the pixels meeting the threshold condition.
    """
    if threshold_type == "greater":
        mask = da > threshold
    elif threshold_type == "less":
        mask = da < threshold
    else:
        raise ValueError("threshold_type must be either 'greater' or 'less'")

    return mask.sum().item() * pixel_area

In [13]:
dnbr_threshold = 0.27
sensitivity = 0.01

# Pixel area in km² (calculated from the raster resolution, which is in metres)
x_res, y_res = dnbr.rio.resolution()
pixel_area = abs(x_res * y_res) / 1e6

records = []

for t in dnbr["time"].values:
    dnbr_t = dnbr.sel(time=t)
    records.append({
        "date": pd.Timestamp(t).date(),
        "area_burned_km2": calculate_thresholded_area(dnbr_t, dnbr_threshold, pixel_area),
        # A higher threshold gives a smaller (lower-bound) area, and vice versa
        "area_burned_lower_km2": calculate_thresholded_area(dnbr_t, dnbr_threshold + sensitivity, pixel_area),
        "area_burned_upper_km2": calculate_thresholded_area(dnbr_t, dnbr_threshold - sensitivity, pixel_area),
    })

df_area_burned = pd.DataFrame(records).round(3)
df_area_burned.to_csv(timeseries_dir / "burned_area.csv", index=False)

df_area_burned

,date,area_burned_km2,area_burned_lower_km2,area_burned_upper_km2
0,2026-04-22,0.000,0.000,0.000
1,2026-04-29,32.548,30.273,34.786
2,2026-04-30,41.962,39.385,44.469
3,2026-05-24,2.902,2.696,3.118


## dNBR within the fire perimeter through time

Alternative measure: calculate the mean and standard deviation of dNBR within the EFFIS fire perimeter on each date.

In [15]:
dnbr_clipped = dnbr.rio.clip(fire_gdf.geometry, fire_gdf.crs, drop=True)

df_dnbr_stats = pd.DataFrame({
    "date": pd.to_datetime(dnbr_clipped["time"].values).date,
    "dnbr_mean": dnbr_clipped.mean(dim=["x", "y"]).values,
    "dnbr_std": dnbr_clipped.std(dim=["x", "y"]).values,
}).round(4)
df_dnbr_stats.to_csv(timeseries_dir / "dnbr_zonal_stats.csv", index=False)

df_dnbr_stats

,date,dnbr_mean,dnbr_std
0,2026-04-22,0.0000,0.0000
1,2026-04-29,0.2088,0.1731
2,2026-04-30,0.2750,0.1575
3,2026-05-24,0.0714,0.1034


## Variability in NBR across all dates

Finally, calculate the standard deviation of NBR across all four dates.

In [17]:
nbr_std = nbr.std(dim="time")
nbr_std.name = "nbr_std"
nbr_std.attrs = {"long_name": "NBR standard deviation"}

filepath = nbr_dir / f"{aoi_name}_nbr_std.tif"
nbr_std.rio.to_raster(filepath, compress="deflate", dtype="float32")
print(f"Saved {filepath}")

Saved ../data/nbr/powys_nbr_std.tif
